# 01 — Introduction

**Does the text of a company's annual report tell us how risky its shares will be over the next year, beyond what last year's risk already tells us?**

We answer this with about 27,000 US annual reports (Form 10-K) filed between 1996 and 2006. A pre-trained financial language model (FinBERT) reads each report, and a neural network we write and train ourselves in PyTorch turns what it read into a prediction of next year's share-price volatility. This notebook sets out the question, the earlier work it builds on, our approach, and how the report is organised.

## 1. Why this question

Every company listed in the US must file a **10-K** with the Securities and Exchange Commission (SEC) each year. It is long (the median report in our data runs to about 12,000 words across the two sections we use) and it is written partly to warn investors about what could go wrong. If that warning carries real information, a model that reads the report should predict risk better than one that does not.

We measure risk by **volatility**: the standard deviation of a share's daily returns over the 12 months after the report is filed. It is the standard measure of investment risk, and it is easier to predict than the direction of the price, because volatility persists. Calm stocks tend to stay calm and turbulent ones turbulent, the effect formalised by ARCH models (Engle, 1982). So **last year's volatility is a strong baseline**, and the real question is whether the text adds anything on top of it. Following Kogan et al. (2009), we predict the log of volatility and score models by mean squared error (MSE) on it.

## 2. What earlier work found

**Word counts.** Kogan et al. (2009) built the dataset we use. They took Item 7 of each 10-K, *Management's Discussion and Analysis* (MD&A), because it is "where the most important forward-looking content is most likely to be found", and fitted support vector regression on word frequencies. Text on its own was worse than last year's volatility, but text *plus* last year's volatility beat the baseline in every test year from 2001 to 2006, significantly in most (for 2006, an MSE of 0.1402 against 0.1463). Because they published the data and their scores, we can reproduce their baseline and compare our models with theirs directly.

**Finance word lists.** Loughran and McDonald (2011) showed that general-purpose sentiment dictionaries misread financial text (words such as *liability* or *tax* are not negative in a 10-K). They built finance-specific word lists whose counts relate to returns, trading volume and later volatility. Their survey (Loughran & McDonald, 2016) reviews this literature, and Li (2008) showed that the length and readability of annual reports are themselves linked to firm performance. Rekabsaz et al. (2017) extended the volatility task with sentiment features expanded through word embeddings.

**Pre-trained language models.** BERT (Devlin et al., 2019) learns general language from huge amounts of text and can then be reused for a new task: *transfer learning*. FinBERT (Araci, 2019) is BERT further trained on financial news and tuned for financial sentiment; it is the model we use. A second FinBERT (Yang et al., 2020) was trained on 10-Ks, earnings calls and analyst reports, and we compare the two.

**Long documents.** BERT-type models read at most 512 tokens (about 400 words) at a time, far shorter than a 10-K. The usual answer is to cut the document into chunks, encode each chunk, and combine the chunk vectors with a second model (Pappagari et al., 2019). FETILDA (Xia et al., 2022) does this for 10-Ks and volatility prediction. We combine the chunks with **attention pooling**, which learns a weight for each chunk, following attention-based multiple-instance learning (Ilse et al., 2018). A report is a "bag" of chunks, and only some of them may say anything about risk.

## 3. Our approach

```
Annual report: Item 1 (business) + Item 7 (MD&A)
        │
        ▼
 Cut into chunks of 510 tokens (median 32 per report)
        │
        ▼
┌─────────────────────┐
│ FinBERT             │  pre-trained, not changed by us
│ (reads each chunk)  │  → one vector of 768 numbers per chunk
└─────────────────────┘
        │
        ▼
 chunk vectors + last year's volatility
        │
        ▼
┌─────────────────────┐
│ Our network         │  written and trained by us in PyTorch:
│ attention pooling   │  weights the chunks, combines them,
│ + regression head   │  and predicts log volatility
└─────────────────────┘
        │
        ▼
 Predicted log volatility for the next 12 months
```

We compare this main model against a ladder of simpler ones, so we can see where any gain comes from:

| Model | What it tests |
|---|---|
| Copy last year's volatility; straight line on it | The baseline every model must beat |
| Word counts (TF-IDF) + support vector regression | Reproduces Kogan et al. (2009) |
| Finance word lists (Loughran & McDonald, 2011) | Simple, explainable text features |
| Small neural network on word counts | A neural network without FinBERT |
| FinBERT + averaging the chunks | Is FinBERT's reading useful at all? |
| **FinBERT + our attention network** | **Main model**: does learning which chunks matter help? |
| Second FinBERT (Yang et al., 2020) | Which pre-trained reader is better? |

We also run the main models on **Item 1 only, Item 7 only, and both**, to ask which part of the report carries the risk information. Kogan et al. used only Item 7. Item 1, the business description (what the company does, its markets and its competitors), may say as much about risk, and the attention network can show which section it relies on.

Every model is trained on earlier years and tested on a later year, never the other way round, so no model sees the future.

## 4. How this answers the brief

- **(a) A scientific question:** whether report text predicts risk beyond past risk, and which section of the report matters.
- **(b) How the model performs computationally:** we time FinBERT on CPU and GPU, across batch sizes and data sizes. Reading every report once took about 11 hours on a laptop GPU, which is why it is done once and saved.
- **(c) Applying it:** the models are scored on held-out years with confidence intervals, and the attention weights show which chunks the network relies on.
- **1000× more data:** each chunk is encoded independently, so the expensive step splits across many GPUs (map) and the results are combined per report (reduce), the MapReduce pattern (Dean & Ghemawat, 2004). 1000× this corpus is more 10-Ks than exist, so the argument extends to a wider stream: all SEC filing types, earnings-call transcripts and news.

## 5. How the report is organised

Read the notebooks in order:

1. **01-Introduction** (this notebook)
2. **02-Data**: where the data come from, how the two sources are joined, how we checked the labels, and how the text becomes model input
3. *Models, results, scale and conclusion: to follow*

Each member's working, including dead ends, is in their own folder (`Oscar/`, `Alex/`, `Louis/`).

## References

- Araci, D. (2019). FinBERT: Financial sentiment analysis with pre-trained language models. *arXiv:1908.10063*.
- Dean, J. & Ghemawat, S. (2004). MapReduce: Simplified data processing on large clusters. *OSDI 2004*.
- Devlin, J., Chang, M.-W., Lee, K. & Toutanova, K. (2019). BERT: Pre-training of deep bidirectional transformers for language understanding. *NAACL-HLT 2019*, 4171–4186.
- Engle, R. F. (1982). Autoregressive conditional heteroscedasticity with estimates of the variance of United Kingdom inflation. *Econometrica*, 50(4), 987–1007.
- Ilse, M., Tomczak, J. M. & Welling, M. (2018). Attention-based deep multiple instance learning. *ICML 2018*.
- Kogan, S., Levin, D., Routledge, B. R., Sagi, J. S. & Smith, N. A. (2009). Predicting risk from financial reports with regression. *NAACL-HLT 2009*, 272–280. https://aclanthology.org/N09-1031/
- Li, F. (2008). Annual report readability, current earnings, and earnings persistence. *Journal of Accounting and Economics*, 45(2–3), 221–247.
- Loughran, T. & McDonald, B. (2011). When is a liability not a liability? Textual analysis, dictionaries, and 10-Ks. *Journal of Finance*, 66(1), 35–65.
- Loughran, T. & McDonald, B. (2016). Textual analysis in accounting and finance: A survey. *Journal of Accounting Research*, 54(4), 1187–1230.
- Pappagari, R., Żelasko, P., Villalba, J., Carmiel, Y. & Dehak, N. (2019). Hierarchical transformers for long document classification. *IEEE ASRU 2019*.
- Rekabsaz, N., Lupu, M., Baklanov, A., Hanbury, A., Dür, A. & Anderson, L. (2017). Volatility prediction using financial disclosures sentiment with word embedding-based IR models. *ACL 2017*.
- Xia, B., Rawte, V. D., Zaki, M. J. & Gupta, A. (2022). FETILDA: An effective framework for fin-tuned embeddings for long financial text documents. *arXiv:2206.06952*.
- Yang, Y., Uy, M. C. S. & Huang, A. (2020). FinBERT: A pretrained language model for financial communications. *arXiv:2006.08097*.